In [ ]:
# ----------  IMPORTS & SMALL UTILITIES ----------
import os, sys, math, copy, types, logging, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import forward as forward      
import af_2D as af             
import utility_2D as util      




# Device / torch defaults
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_dtype(torch.float32)
torch.backends.cudnn.benchmark = True
print(f"Device: {device} | Python: {sys.version.split()[0]} | Torch: {torch.__version__}")

# ---------- helpers ----------
def to_numpy(x: torch.Tensor):
    return x.detach().cpu().numpy()

def complex_norm_fro(x: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(x.reshape(-1), ord=2)

def complex_rand(shape, dtype=torch.complex64, device=device):
    a = torch.rand(shape, device=device)
    b = torch.rand(shape, device=device)
    return (a + 1j * b).to(dtype)


def crop_window_odd(window: torch.Tensor, cx: float, cy: float, size_odd: int,
                    pad_mode: str = 'reflect', subpixel_align: bool = True):
    """Crop a 2D probe/window around (cx,cy) to an ODD size (e.g., 151)."""
    assert size_odd % 2 == 1, "size_odd must be odd"
    H, W = window.shape
    half = size_odd // 2

    if subpixel_align:
        cx0 = (W - 1) / 2.0
        cy0 = (H - 1) / 2.0
        dx = cx - cx0
        dy = cy - cy0
        window = fourier_shift_image(window, dx, dy)

    cx_i = int(round(cx)); cy_i = int(round(cy))
    x0, y0 = cx_i - half, cy_i - half
    x1, y1 = x0 + size_odd, y0 + size_odd

    pad_left   = max(0, -x0)
    pad_top    = max(0, -y0)
    pad_right  = max(0, x1 - W)
    pad_bottom = max(0, y1 - H)

    if any((pad_left, pad_top, pad_right, pad_bottom)):
        wpad = F.pad(window[None, None, ...],
                     (pad_left, pad_right, pad_top, pad_bottom),
                     mode=pad_mode).squeeze(0).squeeze(0)
        x0 += pad_left; x1 += pad_left; y0 += pad_top; y1 += pad_top
    else:
        wpad = window

    cropped = wpad[y0:y1, x0:x1]
    return cropped, (x0 - pad_left, y0 - pad_top, x1 - pad_left, y1 - pad_top)

def pacbed_from_hwn(cbeds_hwn: np.ndarray) -> np.ndarray:
    """cbeds_hwn: (H,W,N) -> PACBED (H,W)."""
    return cbeds_hwn.astype(np.float32).sum(axis=2)

def find_center_via_hough_skimage(pacbed, r_min=10, r_max=60, r_step=1,
                                  canny_sigma=2.0, low=0.05, high=0.15):
    """Find BF disk: Hough -> binary mask -> COM (returns (cx,cy,r), mask, (cx_com,cy_com), edges)."""
    from skimage.feature import canny
    from skimage.transform import hough_circle, hough_circle_peaks
    from scipy.ndimage import center_of_mass

    img = pacbed.astype(np.float32)
    img = (img - img.min()) / (np.ptp(img) + 1e-12)

    edges = canny(img, sigma=canny_sigma, low_threshold=low, high_threshold=high)
    radii = np.arange(int(r_min), int(r_max) + 1, int(r_step), dtype=int)
    hough_res = hough_circle(edges, radii)
    accums, cx_list, cy_list, r_list = hough_circle_peaks(hough_res, radii, total_num_peaks=1)
    if len(cx_list) == 0:
        raise RuntimeError("No circle detected. Adjust r_min/r_max and thresholds.")

    cx, cy, r = float(cx_list[0]), float(cy_list[0]), int(r_list[0])
    yy, xx = np.ogrid[:img.shape[0], :img.shape[1]]
    mask = (xx - cx)**2 + (yy - cy)**2 <= r**2
    cy_com, cx_com = center_of_mass(mask.astype(np.float32))
    return (cx, cy, r), mask.astype(bool), (float(cx_com), float(cy_com)), edges

def crop_hwn_odd(cbeds_hwn: np.ndarray, cx: float, cy: float, size_odd: int,
                 pad_mode: str = 'reflect'):
    """Crop (H,W,N) stack to an odd size around (cx,cy)."""
    assert size_odd % 2 == 1, "size_odd must be odd"
    H, W, N = cbeds_hwn.shape
    half = size_odd // 2
    cx_i, cy_i = int(round(cx)), int(round(cy))
    x0, y0 = cx_i - half, cy_i - half
    x1, y1 = x0 + size_odd, y0 + size_odd

    pad_left   = max(0, -x0)
    pad_top    = max(0, -y0)
    pad_right  = max(0, x1 - W)
    pad_bottom = max(0, y1 - H)

    arr = cbeds_hwn
    if any((pad_left, pad_top, pad_right, pad_bottom)):
        pad_width = ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0))
        arr = np.pad(arr, pad_width, mode=pad_mode)
        x0 += pad_left; x1 += pad_left; y0 += pad_top; y1 += pad_top

    cropped = arr[y0:y1, x0:x1, :]
    return cropped.astype(cbeds_hwn.dtype, copy=False), (x0 - pad_left, y0 - pad_top, x1 - pad_left, y1 - pad_top)

def show_object(obj: torch.Tensor, title_prefix="Object"):
    amp = torch.abs(obj)
    ph  = torch.angle(obj)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    im0 = ax[0].imshow(to_numpy(amp)); ax[0].set_title(f"{title_prefix} |amp|"); ax[0].axis("off"); fig.colorbar(im0, ax=ax[0])
    im1 = ax[1].imshow(to_numpy(ph), cmap="turbo"); ax[1].set_title(f"{title_prefix} angle"); ax[1].axis("off"); fig.colorbar(im1, ax=ax[1])
    plt.tight_layout(); plt.show()

DATA_PATH = "your_data.npz"
KEY = "cbeds"
raw = np.load(DATA_PATH)[KEY]
print("Raw shape:", raw.shape)

DATA_LAYOUT = "scan_first"   # "scan_first"=(Sx,Sy,kx,ky), "detector_first"=(kx,ky,Sx,Sy), or "hwn"

if raw.ndim == 4:
    if DATA_LAYOUT == "scan_first":
        Sx, Sy, Kx, Ky = raw.shape
        msrmts = raw.transpose(2, 3, 0, 1).reshape(Kx, Ky, -1)
    elif DATA_LAYOUT == "detector_first":
        Kx, Ky, Sx, Sy = raw.shape
        msrmts = raw.reshape(Kx, Ky, -1)
    else:
        raise ValueError(f"4D data but DATA_LAYOUT={DATA_LAYOUT!r}")
    assert Sx == Sy, f"Scan grid not square: {Sx}x{Sy}"
elif raw.ndim == 3:
    assert DATA_LAYOUT == "hwn", "3D data: set DATA_LAYOUT='hwn'"
    msrmts = raw
else:
    raise ValueError(f"Unexpected ndim {raw.ndim}")

# hard validation instead of heuristics:
assert msrmts.shape[0] == msrmts.shape[1], f"Detector not square: {msrmts.shape[:2]}"
n = msrmts.shape[2]
assert round(n**0.5)**2 == n, f"N={n} is not a perfect square — wrong axes flattened?"
msrmts = np.ascontiguousarray(msrmts)
print("Normalized (kx, ky, N):", msrmts.shape)
# ------------------- Example usage (HWN -> HWN) -------------------
# msrmts: your input array shaped (H, W, N) = (152, 152, 5776)
H, W, N = msrmts.shape
print("Input HWN:", msrmts.shape)

#shift defines the sampling so shift = 2 means your resulting pixel size will be 2* x and y size of the 4D-dataset
shift = 2

# PACBED & Hough
pacbed = pacbed_from_hwn(msrmts)
(cx, cy, r), mask, (cx_com, cy_com), edges = find_center_via_hough_skimage(
    pacbed, r_min=10, r_max=60, r_step=1, canny_sigma=2.0, low=0.05, high=0.15
)
print(f"Hough center: ({cx:.1f}, {cy:.1f}), r={r}")
print(f"Disc COM from mask: ({cx_com:.2f}, {cy_com:.2f})")

# Visual sanity check
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(pacbed); ax[0].set_title("PACBED")
circ = plt.Circle((cx, cy), r, fill=False, lw=2); ax[0].add_patch(circ)
ax[1].imshow(edges); ax[1].set_title("Canny edges")
ax[2].imshow(mask); ax[2].set_title("Disc mask")
for a in ax:
    a.scatter([cx_com], [cy_com], s=25); a.set_axis_off()
plt.show()

# Choose an odd crop size (keep it <= min(H,W)); e.g., 151 or based on r+margin
crop_size_odd = min(H, W) - 1  

cropped_hwn, (x0, y0, x1, y1) = crop_hwn_odd(msrmts, cx_com, cy_com, crop_size_odd, pad_mode='reflect')
print("Cropped HWN:", cropped_hwn.shape)  # (Hc, Wc, N)

# ---
b2 = torch.from_numpy(cropped_hwn) 
print(b2.shape)
    
# ---- Gaussian initial ----
delta = crop_size_odd
c = (delta - 1) / 2.0
yy, xx = np.meshgrid(np.arange(delta), np.arange(delta), indexing='ij')
r2 = (xx - c)**2 + (yy - c)**2

hwhm = 20.0                                  # half max at radius 20 px
sigma = hwhm / np.sqrt(2.0 * np.log(2.0))    # ≈ 16.99 px
sigma2 = sigma**2

window_np = np.exp(-0.5 * r2 / sigma2)

window = torch.from_numpy(window_np).to(device).to(torch.complex64)
window = window / torch.abs(window).max()


plt.imshow(np.abs(window.cpu())); plt.title("|window| (centered)"); plt.show()

# move center -> (0,0) corner for FFT convention
cy0, cx0 = delta // 2, delta // 2
window = torch.roll(window, shifts=(-cy0, -cx0), dims=(0, 1))


loc_num = round(b2.shape[2] ** 0.5)
d =loc_num * shift + b2.shape[0] - shift
d_orig = loc_num
b2 = b2.to(device)
assert loc_num * loc_num == msrmts.shape[2], "Scan grid is not square"
b2 = b2.to(device)
window = window.to(device)
print(window.shape)



# === Your original call pattern begins here ===

# ptycho model (Torch forward)
par = forward.ptycho(
    object_shape=(int(d), int(d)),
    window=window,
    float_shift=False,
    circular=False,
    loc_type='grid',
    shift=shift,
    fourier_dimension=(b2.shape[0], b2.shape[1])
)


# AG parameter namespaces
AG_pars = types.SimpleNamespace(enable_AG=True, control=0.4, tau=0.4, AG_iterations=2)
AG_pars_shift = types.SimpleNamespace(enable_AG=False, control=0.5, tau=0.1, AG_iterations=2)

# Torch-only initialization that mirrors your NumPy scaling idea
obj_r = complex_rand((int(d), int(d)), device=device)
scale = torch.sqrt(torch.sum(b2)) / complex_norm_fro(window).clamp_min(1e-12)
obj_r = obj_r / complex_norm_fro(obj_r).clamp_min(1e-12) * scale
win_r = window.clone()

# Blind Amplitude Flow
baf = af.blind_amplitude_flow(
    measurements=b2,
    ptycho=par,
    number_of_iterations=10,
    object_subiterations=5,
    window_subiterations=5,
    shift_subiterations=100,
    skip_win_it=5,
    update_shifts=False,
    skip_shift_it=15,
    shift_learning_rate_type='custom',
    shift_learning_rate=1.0,
    update_shifts_period=5,
    grad_thr_object=1e-5,
    grad_thr_window=1e-5,
    AGP_object=AG_pars,
    AGP_window=AG_pars,
    AGP_shift=AG_pars_shift,
    epsilon=1e-12,
    alpha_ST=10e-3, 
    alpha_TV=10e-3,
    TV_param_obj=1.0,
    TV_param_win=1.0,
    reweight=True,
    verbose=1,
    track=True,
    obj_tr=obj_r.clone(),
    win_tr=window.clone(),
    track_it=torch.tensor([10, 20, 50])
)

start_time = time.time()
obj_r, win_r, pl_value = baf.run(obj_r, win_r)
end_time = time.time()
print('Time (s): ', end_time - start_time)

# Visualize reconstructions
show_object(obj_r, title_prefix="Reconstructed obj")
show_object(win_r, title_prefix="Reconstructed win")

# Forward check with updated window and relative measurement error
par_r = par.copy()
par_r.set_window(win_r)

f_r2 = par_r.forward_2D_pty(obj_r)
b_r2 = par_r.forward_to_meas_2D_pty(f_r2)
print('Relative measurement error: ', util.relative_sq_measurement_error(b2, b_r2))

print("b2 shape:", tuple(b2.shape))



# ---------- CENTERED ZOOM SERIES (full → +10 → +20) ----------

# Show full reconstruction first
show_object(obj_r,  title_prefix="Reconstructed obj (full)")
obj_r2, win_r2 = util.eliminate_linear_ambiguity(obj_r, win_r, threshold=1e-1)
show_object(obj_r2, title_prefix="Reconstructed obj (post-ambiguity, full)")

# ------- ZOOM INTO CENTER WITH show_object ONLY (0 → 10 → 20) -------

def center_crop(t: torch.Tensor, half: int):
    h, w = t.shape
    cx, cy = h // 2, w // 2
    x0, x1 = max(0, cx - half), min(h, cx + half)
    y0, y1 = max(0, cy - half), min(w, cy + half)
    return t[x0:x1, y0:y1]

# Start from the largest sensible half-size, then shrink to zoom
h, w = obj_r.shape
base_half = min(h, w) // 2   # full field (centered)

# how much to shrink each step (pixels)
for shrink in (0, 90, 100):
    half = max(8, base_half - shrink)  # keep at least an 8×8 window
    crop = center_crop(obj_r, half)
    show_object(crop, title_prefix=f"Reconstructed obj (center ±{half})")
